In [1]:
# Firstup, combined features predicitive model

#This has already been done will use past version

# Next, genes only predicitive model feature importances

# done already

Processing: Ast
Processing: Mic
Processing: In
Processing: Oli
Processing: Opc
Processing: Ex


/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/openpyxl/workbook/child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")


Sheet 'AD_Combined_feature_Importances_CellLevel' written to /n/groups/patel/adithya/TriSCOPE_tables.xlsx


In [1]:
# The first table that needs to be redone is Supplemental Table 7: All_NonGWAS_Fisher_TEsts.xlsx. Specifically:
# I need to replace the DEG results reference for both CERAD and AD to the random effects folders
import os
import joblib
import pandas as pd
import numpy as np
from collections import defaultdict
from scipy.stats import fisher_exact
from statsmodels.stats.contingency_tables import Table2x2
# reran with fdr correction

# --- BH helper ---
def bh_fdr(pvals):
    """Benjamini–Hochberg FDR correction."""
    p = np.asarray(pvals, dtype=float)
    n = p.size
    if n == 0:
        return p
    order = np.argsort(p)
    ranks = np.arange(1, n + 1)
    p_sorted = p[order]
    q_sorted = p_sorted * n / ranks
    q_sorted = np.minimum.accumulate(q_sorted[::-1])[::-1]  # enforce monotonicity
    q = np.empty_like(q_sorted)
    q[order.argsort()] = np.minimum(q_sorted, 1.0)
    return q

# === Background genes ===
gene_matrix = pd.read_parquet('/home/adm808/NormalizedCellMatrixSyn18485175.parquet')
background_genes = set(gene_matrix.index.str.upper())
print(f"Total background genes: {len(background_genes)}")

# === Cell type labels ===
ct_labels = {
    'Ast': 'Astrocytes',
    'Mic': 'Microglia',
    'In': 'Inhibitory Neurons',
    'Oli': 'Oligodendrocytes',
    'Opc': 'Oligodendrocyte Progenitor Cells',
    'Ex': 'Excitatory Neurons'
}
cell_types = list(ct_labels.keys())

# === Paths ===
ad_base_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_new"
cerad_base_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_cerad"
de_base_dir = "/n/scratch/users/a/adm808/Revision/Clincal_batch_DE_Outputs_revision"
cerad_de_base_dir = "/n/scratch/users/a/adm808/Revision/Cerad_batch_DE_Outputs_revision"

de_files = {
    "Ast": "poisson_DE_results_Ast.csv",
    "Mic": "poisson_DE_results_Mic.csv",
    "In": "poisson_DE_results_In.csv",
    "Oli": "poisson_DE_results_Oli.csv",
    "Opc": "poisson_DE_results_Opc.csv",
    "Ex": "poisson_DE_results_Ex.csv"
}

# === Collect results for all cell types/tests ===
all_results = []

# === Prepare Excel writer ===
output_excel_path = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/All_NonGWAS_Fisher_Tests.xlsx"
with pd.ExcelWriter(output_excel_path, engine='xlsxwriter') as writer:

    # --- Title page ---
    title_text = (
        "This workbook summarizes all non-GWAS Fisher tests run per cell type.\n\n"
        "Included tests:\n"
        "1. AD predictors vs CERAD predictors\n"
        "2. AD predictors vs AD DE genes\n"
        "3. CERAD predictors vs CERAD DE genes\n"
        "4. AD DE genes vs CERAD DE genes\n\n"
        "QValue is Benjamini–Hochberg FDR, calculated separately for each test type across all 6 cell types."
    )
    pd.DataFrame({"Description": [title_text]}).to_excel(writer, sheet_name="Title_Page", index=False)

    for cell_short in cell_types:
        cell_name = ct_labels[cell_short]
        rows = []

        # === Get AD predictors ===
        gene_presence_ad = defaultdict(int)
        for split in range(1, 6):
            path = os.path.join(ad_base_dir, cell_short, f"split_{split}", "maximal_classifier.joblib")
            if os.path.exists(path):
                model = joblib.load(path)
                for gene, imp in zip(model.feature_names_in_, model.feature_importances_):
                    if imp > 0:
                        gene_presence_ad[gene.upper()] += 1
        ad_predictors = {g for g, cnt in gene_presence_ad.items() if cnt >= 2}

        # === Get CERAD predictors ===
        gene_presence_cerad = defaultdict(int)
        for split in range(1, 6):
            path = os.path.join(cerad_base_dir, cell_short, f"split_{split}", "maximal_classifier.joblib")
            if os.path.exists(path):
                model = joblib.load(path)
                for gene, imp in zip(model.feature_names_in_, model.feature_importances_):
                    if imp > 0:
                        gene_presence_cerad[gene.upper()] += 1
        cerad_predictors = {g for g, cnt in gene_presence_cerad.items() if cnt >= 2}

        # === Get AD DE genes ===
        # ad_de_df = pd.read_csv(os.path.join(de_base_dir, de_files[cell_short]))
        # ad_de_df["gene_upper"] = ad_de_df["gene"].str.upper()
        # ad_de_genes = set(ad_de_df[ad_de_df["p_adj"] < 0.05]["gene_upper"])

        # # === Get CERAD DE genes ===
        # cerad_de_df = pd.read_csv(os.path.join(cerad_de_base_dir, de_files[cell_short]))
        # cerad_de_df["gene_upper"] = cerad_de_df["gene"].str.upper()
        # cerad_de_genes = set(cerad_de_df[cerad_de_df["p_adj"] < 0.05]["gene_upper"])

                # === Get AD DE genes ===
        ad_de_df = pd.read_csv(os.path.join(de_base_dir, de_files[cell_short]))
        ad_de_df["gene_upper"] = ad_de_df["gene"].str.upper()
        ad_de_genes = set(
            ad_de_df[(ad_de_df["p_adj"] < 0.05) & (ad_de_df["log2FC"].abs() > 0.25)]["gene_upper"]
        )

        # === Get CERAD DE genes ===
        cerad_de_df = pd.read_csv(os.path.join(cerad_de_base_dir, de_files[cell_short]))
        cerad_de_df["gene_upper"] = cerad_de_df["gene"].str.upper()
        cerad_de_genes = set(
            cerad_de_df[(cerad_de_df["p_adj"] < 0.05) & (cerad_de_df["log2FC"].abs() > 0.25)]["gene_upper"]
        )

        # --- Define helper for adding results ---
        def add_test(test_name, set1, set2):
            overlap = set1 & set2
            a = len(overlap)
            b = len(set1 - overlap)
            c = len(set2 - overlap)
            d = len(background_genes - (set1 | set2))
            table = [[a, b], [c, d]]
            or_val, p_val = fisher_exact(table, alternative='greater')
            ci_low, ci_high = Table2x2(table).oddsratio_confint()
            row = {
                "CellType": cell_name,
                "Test": test_name,
                "Num_Set1_Genes": len(set1),
                "Num_Set2_Genes": len(set2),
                "Overlap": a,
                "OddsRatio": or_val,
                "CI_Low": ci_low,
                "CI_High": ci_high,
                "PValue": p_val,
                "Overlap_Genes": ", ".join(sorted(overlap))
            }
            rows.append(row)
            all_results.append(row)

        # --- Run the 4 tests ---
        add_test("AD predictors vs CERAD predictors", ad_predictors, cerad_predictors)
        add_test("AD predictors vs AD DE genes", ad_predictors, ad_de_genes)
        add_test("CERAD predictors vs CERAD DE genes", cerad_predictors, cerad_de_genes)
        add_test("AD DE genes vs CERAD DE genes", ad_de_genes, cerad_de_genes)

        # Save sheet (QValues will be added later after FDR)
        pd.DataFrame(rows).to_excel(writer, sheet_name=cell_name[:31], index=False)

    # === FDR correction per test type ===
    all_df = pd.DataFrame(all_results)
    all_df["QValue"] = all_df.groupby("Test")["PValue"].transform(lambda p: bh_fdr(p.values))

    # Write updated per-cell sheets with QValue
    for cell_name in all_df["CellType"].unique():
        df_cell = all_df[all_df["CellType"] == cell_name].copy()
        df_cell.to_excel(writer, sheet_name=cell_name[:31], index=False)

    # Optional: summary sheet with all results
    all_df.to_excel(writer, sheet_name="All_Tests_Summary", index=False)

print(f"Workbook saved to: {output_excel_path}")

Total background genes: 17926


/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.5.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.5.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator ColumnTransformer from version 1.5.1 w

Workbook saved to: /n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/All_NonGWAS_Fisher_Tests.xlsx


In [ ]:

# AD based differential expression first
import pandas as pd
import os
#reran aug 11th

# Base DE files directory
# base_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Final_Outputs_Figures/Differential_Expression_Final/Fixed"
base_dir = "/n/scratch/users/a/adm808/Revision/Clincal_batch_DE_Outputs_revision"
files = [
    "poisson_DE_results_In.csv",
    "poisson_DE_results_Mic.csv",
    "poisson_DE_results_Oli.csv",
    "poisson_DE_results_Opc.csv",
    "poisson_DE_results_Ast.csv",
    "poisson_DE_results_Ex.csv"
]

# Map short codes to pretty names
ct_labels = {
    'Ast': 'Astrocytes',
    'Mic': 'Microglia',
    'In': 'Inhibitory Neurons',
    'Oli': 'Oligodendrocytes',
    'Opc': 'Oligodendrocyte Progenitor Cells',
    'Ex': 'Excitatory Neurons'
}

# Output workbook path
out_path = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/AD_Differential_Expression.xlsx"

# Create an Excel writer
with pd.ExcelWriter(out_path, engine='openpyxl') as writer:
    # Add title page
    title_text = (
        "This workbook contains the full differential expression analysis results between Alzheimer's samples and controls."
        "Poisson model for Alzheimer's disease vs. Control across major brain cell types. "
        "Sheets correspond to individual cell types."
    )
    title_df = pd.DataFrame({"Description": [title_text]})
    title_df.to_excel(writer, sheet_name="Title_Page", index=False)
    
    # Add each cell type sheet
    for file in files:
        cell_short = file.replace("poisson_DE_results_", "").replace(".csv", "")
        cell_name = ct_labels[cell_short]
        
        # Load data
        df = pd.read_csv(os.path.join(base_dir, file))

        df["DEG"] = (df["p_adj"] < 0.05)
        
        # Write to sheet
        df.to_excel(writer, sheet_name=cell_name, index=False)

print(f"Workbook saved to: {out_path}")

/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/openpyxl/workbook/child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")


Workbook saved to: /n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/AD_Differential_Expression.xlsx


In [9]:
# AD based differential expression first
import pandas as pd
import os
# reran aug 11th

# Base DE files directory
# base_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Final_Outputs_Figures/Differential_Expression_Final/CERAD"
base_dir = "/n/scratch/users/a/adm808/Revision/Cerad_batch_DE_Outputs_revision"
files = [
    "poisson_DE_results_In.csv",
    "poisson_DE_results_Mic.csv",
    "poisson_DE_results_Oli.csv",
    "poisson_DE_results_Opc.csv",
    "poisson_DE_results_Ast.csv",
    "poisson_DE_results_Ex.csv"
]

# Map short codes to pretty names
ct_labels = {
    'Ast': 'Astrocytes',
    'Mic': 'Microglia',
    'In': 'Inhibitory Neurons',
    'Oli': 'Oligodendrocytes',
    'Opc': 'Oligodendrocyte Progenitor Cells',
    'Ex': 'Excitatory Neurons'
}

# Output workbook path
out_path = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/CERAD_Differential_Expression.xlsx"

# Create an Excel writer
with pd.ExcelWriter(out_path, engine='openpyxl') as writer:
    # Add title page
    title_text = (
        "This workbook contains the full differential expression analysis results between AD-like CERAD pathology samples and controls."
        "Poisson model for Alzheimer's disease vs. Control (based on CERAD pathology) across major brain cell types. "
        "Sheets correspond to individual cell types."
    )
    title_df = pd.DataFrame({"Description": [title_text]})
    title_df.to_excel(writer, sheet_name="CERAD_Title_Page", index=False)
    
    # Add each cell type sheet
    for file in files:
        cell_short = file.replace("poisson_DE_results_", "").replace(".csv", "")
        cell_name = ct_labels[cell_short]
        
        # Load data
        df = pd.read_csv(os.path.join(base_dir, file))

        # df["DEG"] = (df["p_adj"] < 0.05)
        df["DEG"] = (df["p_adj"] < 0.05) & (df["log2FC"].abs() > 0.25)
        
        # Write to sheet
        df.to_excel(writer, sheet_name=cell_name, index=False)

print(f"Workbook saved to: {out_path}")

/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/openpyxl/workbook/child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")


Workbook saved to: /n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/CERAD_Differential_Expression.xlsx


In [6]:
# AD based differential expression first
# reran - agu 11th
import pandas as pd
import os

# Base DE files directory
base_dir = "/n/scratch/users/a/adm808/Downloads/Revision/Maximal_Data_DEG_results/Processed"

files = [
    "poisson_DE_results_PFC_Ast_COMBINED.csv",
    "poisson_DE_results_PFC_Mic_COMBINED.csv",
    "poisson_DE_results_PFC_Oli_COMBINED.csv",
    "poisson_DE_results_PFC_Opc_COMBINED.csv",
    "poisson_DE_results_PFC_Ex_COMBINED.csv",
    "poisson_DE_results_PFC_In_COMBINED.csv",
]

# Map short codes to pretty names
ct_labels = {
    'Ast': 'Astrocytes',
    'Mic': 'Microglia',
    'In': 'Inhibitory Neurons',
    'Oli': 'Oligodendrocytes',
    'Ex': 'Excitatory Neurons',
    'Opc': "Oligodendrocytes Progenitor Cells"
}

# Output workbook path
out_path = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/Validation_Differential_Expression.xlsx"

# Create an Excel writer
with pd.ExcelWriter(out_path, engine='openpyxl') as writer:
    # Add title page
    title_text = (
        "This workbook contains the full differential expression analysis results for our validation cohort between Alzheimer's samples and controls."
        "Poisson model for Alzheimer's disease vs. Control across major brain cell types. "
        "Sheets correspond to individual cell types."
    )
    title_df = pd.DataFrame({"Description": [title_text]})
    title_df.to_excel(writer, sheet_name="Title_Page", index=False)
    
    # Add each cell type sheet
    for file in files:
        cell_short = file.split("_")[4]
        # cell_short = file.replace("poisson_DE_results_", "").replace(".csv", "")
        cell_name = ct_labels[cell_short]
        
        # Load data
        df = pd.read_csv(os.path.join(base_dir, file))
        df["DEG"] = (df["p_adj"] < 0.05) & (df["log2FC"].abs() > 0.25)

        # df["DEG"] = (df["p_adj"] < 0.05)
        
        # Write to sheet
        df.to_excel(writer, sheet_name=cell_name, index=False)

print(f"Workbook saved to: {out_path}")

/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/openpyxl/workbook/child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")


Workbook saved to: /n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/Validation_Differential_Expression.xlsx


In [11]:
# redoing supplemental table 11

import os
import joblib
import pandas as pd
import numpy as np
from collections import defaultdict
from scipy.stats import fisher_exact
# reran aug 11th

# === Background genes ===
gene_matrix = pd.read_parquet('/home/adm808/NormalizedCellMatrixSyn18485175.parquet')
background_genes = set(gene_matrix.index.str.upper())
print(f"Total background genes: {len(background_genes)}")

# === Cell type labels ===
ct_labels = {
    'Ast': 'Astrocytes',
    'Mic': 'Microglia',
    'In': 'Inhibitory Neurons',
    'Oli': 'Oligodendrocytes',
    'Ex': 'Excitatory Neurons',
    "Opc": 'Oligodendrocytes Progenitors'
}
cell_types = list(ct_labels.keys())

# === Paths ===
ad_base_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_new"
cerad_base_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Validation_Multirun_cell_on_cell_genes_Lau_rfe"
# de_base_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Final_Outputs_Figures/Differential_Expression_Final/Fixed/"
# cerad_de_base_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Final_Outputs_Figures/Differential_Expression_Final/Validation/"

# AD / clinical DE (non-combined)
de_base_dir = "/n/scratch/users/a/adm808/Revision/Clincal_batch_DE_Outputs_revision"

# CERAD / validation DE (PFC combined)
cerad_de_base_dir = "/n/scratch/users/a/adm808/Downloads/Revision/Maximal_Data_DEG_results/Processed"


# AD DE filenames
ad_de_files = {
    "Ast": "poisson_DE_results_Ast.csv",
    "Mic": "poisson_DE_results_Mic.csv",
    "In":  "poisson_DE_results_In.csv",
    "Oli": "poisson_DE_results_Oli.csv",
    "Opc": "poisson_DE_results_Opc.csv",
    "Ex":  "poisson_DE_results_Ex.csv",
}

# CERAD DE filenames (combined PFC)
cerad_de_files = {
    "Ast": "poisson_DE_results_PFC_Ast_COMBINED.csv",
    "Mic": "poisson_DE_results_PFC_Mic_COMBINED.csv",
    "In":  "poisson_DE_results_PFC_In_COMBINED.csv",
    "Oli": "poisson_DE_results_PFC_Oli_COMBINED.csv",
    "Opc": "poisson_DE_results_PFC_Opc_COMBINED.csv",
    "Ex":  "poisson_DE_results_PFC_Ex_COMBINED.csv",
}

# === Prepare Excel writer ===
output_excel_path = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/Validation_Fisher_Tests.xlsx"
with pd.ExcelWriter(output_excel_path, engine='xlsxwriter') as writer:

    # --- Add title page first ---
    title_text = (
        "This workbook summarizes all Fisher tests for the Validation Cohort run per cell type.\n\n"
        "Included tests:\n"
        "1. Original (AD) predictors (Mathys 2019) vs Validation predictors (Lau 2020 dataset)\n"
        "2. Validation predictors (Lau 2020 dataset) vs Validation DE genes (Mathys 2023 dataset)\n"
        "3. Original (AD) (Mathys 2019) DE genes vs Validation DE genes (Mathys 2023 dataset)"
    )
    title_df = pd.DataFrame({"Description": [title_text]})
    title_df.to_excel(writer, sheet_name="Title_Page", index=False)

    # === Loop through each cell type ===
    for cell_short in cell_types:
        cell_name = ct_labels[cell_short]
        rows = []

        ### === Get AD predictors ===
        gene_presence_ad = defaultdict(int)
        for split in range(1, 6):
            path = os.path.join(ad_base_dir, cell_short, f"split_{split}", "maximal_classifier.joblib")
            if not os.path.exists(path):
                continue
            model = joblib.load(path)
            for gene, imp in zip(model.feature_names_in_, model.feature_importances_):
                if imp > 0:
                    gene_presence_ad[gene.upper()] += 1
        ad_predictors = {gene for gene, count in gene_presence_ad.items() if count >= 2}

        ### === Get Validation predictors ===
        gene_presence_cerad = defaultdict(int)
        for split in range(1, 6):
            path = os.path.join(cerad_base_dir, cell_short, f"split_{split}", "maximal_classifier.joblib")
            if not os.path.exists(path):
                continue
            model = joblib.load(path)
            for gene, imp in zip(model.feature_names_in_, model.feature_importances_):
                if imp > 0:
                    gene_presence_cerad[gene.upper()] += 1
        cerad_predictors = {gene for gene, count in gene_presence_cerad.items() if count >= 2}

        ### === Get AD DE genes ===
        ad_de_df = pd.read_csv(os.path.join(de_base_dir, ad_de_files[cell_short]))
        ad_de_df["gene_upper"] = ad_de_df["gene"].str.upper()
        # ad_de_genes = set(ad_de_df[ad_de_df["p_adj"] < 0.05]["gene_upper"])
        ad_de_genes = set(
            ad_de_df[(ad_de_df["p_adj"] < 0.05) & (ad_de_df["log2FC"].abs() > 0.25)]["gene_upper"]
        )

        ### === Get Validation DE genes ===
        cerad_de_df = pd.read_csv(os.path.join(cerad_de_base_dir, cerad_de_files[cell_short]))
        cerad_de_df["gene_upper"] = cerad_de_df["gene"].str.upper()
        cerad_de_genes = set(
            cerad_de_df[(cerad_de_df["p_adj"] < 0.05) & (cerad_de_df["log2FC"].abs() > 0.25)]["gene_upper"]
        )
        # cerad_de_genes = set(cerad_de_df[cerad_de_df["p_adj"] < 0.05]["gene_upper"])

        # === Test 1: AD predictors vs Validation predictors ===
        overlap = ad_predictors & cerad_predictors
        a = len(overlap)
        b = len(ad_predictors - overlap)
        c = len(cerad_predictors - overlap)
        d = len(background_genes - (ad_predictors | cerad_predictors))
        table = [[a, b], [c, d]]
        or1, pv1 = fisher_exact(table, alternative='greater')
        rows.append({
            "Test": "Original (AD) predictors vs Validation predictors",
            "Num_Set1_Genes": len(ad_predictors),
            "Num_Set2_Genes": len(cerad_predictors),
            "Overlap": a,
            "OddsRatio": or1,
            "PValue": pv1,
            "Overlap_Genes": ", ".join(sorted(overlap))
        })

        # === Test 2: Validation predictors vs Validation DE genes ===
        overlap = cerad_predictors & cerad_de_genes
        a = len(overlap)
        b = len(cerad_predictors - overlap)
        c = len(cerad_de_genes - overlap)
        d = len(background_genes - (cerad_predictors | cerad_de_genes))
        table = [[a, b], [c, d]]
        or2, pv2 = fisher_exact(table, alternative='greater')
        rows.append({
            "Test": "Validation predictors vs Validation DE genes",
            "Num_Set1_Genes": len(cerad_predictors),
            "Num_Set2_Genes": len(cerad_de_genes),
            "Overlap": a,
            "OddsRatio": or2,
            "PValue": pv2,
            "Overlap_Genes": ", ".join(sorted(overlap))
        })

        # === Test 3: Original (AD) DE genes vs Validation DE genes ===
        overlap = ad_de_genes & cerad_de_genes
        a = len(overlap)
        b = len(ad_de_genes - overlap)
        c = len(cerad_de_genes - overlap)
        d = len(background_genes - (ad_de_genes | cerad_de_genes))
        table = [[a, b], [c, d]]
        or3, pv3 = fisher_exact(table, alternative='greater')
        rows.append({
            "Test": "Original (AD) DE genes vs Validation DE genes",
            "Num_Set1_Genes": len(ad_de_genes),
            "Num_Set2_Genes": len(cerad_de_genes),
            "Overlap": a,
            "OddsRatio": or3,
            "PValue": pv3,
            "Overlap_Genes": ", ".join(sorted(overlap))
        })

        # === Save sheet for this cell type ===
        cell_df = pd.DataFrame(rows)
        cell_df.to_excel(writer, sheet_name=cell_name[:31], index=False)

print(f"Workbook saved to: {output_excel_path}")

Total background genes: 17926
Workbook saved to: /n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/Validation_Fisher_Tests.xlsx


In [12]:
import os
import shutil

src = "/n/groups/patel/adithya/Pseudotime_Outputs_HVG_final/publication_summary_statistics.csv"
dst_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/"
os.makedirs(dst_dir, exist_ok=True)

dst = os.path.join(dst_dir, "Pseudotime_summary_statistics.csv")

shutil.copy2(src, dst)

'/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/Pseudotime_summary_statistics.csv'

In [13]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics import roc_curve, auc

# -----------------------------
# Paths
# -----------------------------
BASE_ROOT = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs"

MODEL_DIRS = {
    "Genes_GBM": "Multirun_cell_on_cell_genes_new",
    "Genes_LASSO": "Multirun_cell_on_cell_genes_lasso",
    "Genes_ElasticNet": "Multirun_cell_on_cell_genes_elastic",
    "Genes_RandomForest": "Multirun_cell_on_cell_genes_rf",
    "Genes_Permuted": "Multirun_cell_on_cell_genes_permuted",
}

CELL_TYPES = ["Ast", "Ex", "In", "Oli", "Opc", "Mic"]
SPLITS = range(1, 6)

# -----------------------------
# Collect results
# -----------------------------
rows = []

for cell_type in CELL_TYPES:
    for model_name, model_dir in MODEL_DIRS.items():

        aucs = []

        base_path = os.path.join(BASE_ROOT, model_dir, cell_type)

        for i in SPLITS:
            pred_file = os.path.join(
                base_path, f"split_{i}", "test_predictions.csv"
            )

            if not os.path.exists(pred_file):
                continue

            df = pd.read_csv(pred_file)
            y_true = df["true_label"]
            y_score = df["predicted_proba"]

            fpr, tpr, _ = roc_curve(y_true, y_score)
            aucs.append(auc(fpr, tpr))

        if len(aucs) > 0:
            rows.append({
                "cell_type": cell_type,
                "model": model_name,
                "mean_auc": np.mean(aucs),
                "std_auc": np.std(aucs),
                "n_splits_used": len(aucs),
            })

# -----------------------------
# Final table
# -----------------------------
auc_table = pd.DataFrame(rows)

# Optional: sort nicely
auc_table = auc_table.sort_values(
    ["cell_type", "mean_auc"], ascending=[True, False]
)

# Save
out_path = (
    "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/"
    "Model_benchmarking_auc_summary.csv"
)
auc_table.to_csv(out_path, index=False)

print("Saved:", out_path)
print(auc_table)

Saved: /n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/Model_benchmarking_auc_summary.csv
   cell_type               model  mean_auc   std_auc  n_splits_used
1        Ast         Genes_LASSO  0.589058  0.054525              5
2        Ast    Genes_ElasticNet  0.575045  0.050046              5
0        Ast           Genes_GBM  0.567834  0.064186              5
3        Ast  Genes_RandomForest  0.561477  0.076015              5
4        Ast      Genes_Permuted  0.499498  0.146551              5
6         Ex         Genes_LASSO  0.606992  0.054479              5
7         Ex    Genes_ElasticNet  0.603144  0.053706              5
5         Ex           Genes_GBM  0.529827  0.042344              5
8         Ex  Genes_RandomForest  0.520718  0.075128              5
9         Ex      Genes_Permuted  0.468646  0.176697              5
13        In  Genes_RandomForest  0.655847  0.043110              5
10        In           Genes_GBM  0.627545  0.074425              5
11      

In [14]:
import os
import pandas as pd

# -----------------------------
# Paths
# -----------------------------
in_path = "/n/scratch/users/a/adm808/Revision/Coloc/coloc_results_AD_GWAS_GTEx_v10_Cortex_and_BA9_all_brain.csv"
out_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables"
os.makedirs(out_dir, exist_ok=True)

out_xlsx = os.path.join(out_dir, "Coloc_Results.xlsx")

# -----------------------------
# Load full coloc table (NO modification)
# -----------------------------
df = pd.read_csv(in_path)

# -----------------------------
# Write one sheet per cell type
# -----------------------------
with pd.ExcelWriter(out_xlsx, engine="xlsxwriter") as writer:
    for ct, df_ct in df.groupby("cell_type"):
        sheet_name = str(ct)[:31]  # Excel sheet name limit
        df_ct.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"Saved Excel workbook: {out_xlsx}")

Saved Excel workbook: /n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/Coloc_Results.xlsx


In [16]:
import pandas as pd

DRUGBANK_CSV = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/DrugBank_Predictive_Targets.csv"

df = pd.read_csv(DRUGBANK_CSV)

OUT_PATH = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/DrugBank_Predictive_Target.csv"

df.to_csv(OUT_PATH, index=False)

print("Saved:", OUT_PATH)

Saved: /n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/DrugBank_Predictive_Target.csv


In [1]:
import pandas as pd
from pathlib import Path

# ============================================================
# Input DEG result files (4 total)
# ============================================================
deg_paths = {
    "Mic_Discovery48": "/n/scratch/users/a/adm808/Contrasts/NPvsCAD/Mathys_Old_Data_DEG_results/poisson_DE_results_Mic.csv",
    "Oli_Discovery48": "/n/scratch/users/a/adm808/Contrasts/NPvsCAD/Mathys_Old_Data_DEG_results/poisson_DE_results_Oli.csv",
    "Mic_Validation48": "/n/scratch/users/a/adm808/Contrasts/NPvsCAD/Mathys_New_Data_DEG_results/poisson_DE_results_PFC_Mic.csv",
    "Oli_Validation48": "/n/scratch/users/a/adm808/Contrasts/NPvsCAD/Mathys_New_Data_DEG_results/poisson_DE_results_PFC_Oli.csv",
}

# ============================================================
# Output Excel workbook
# ============================================================
out_dir = Path("/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables")
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / "DE_under_matched_Neuropathology.xlsx"

# ============================================================
# Write to Excel (one sheet per file)
# ============================================================
with pd.ExcelWriter(out_path, engine="xlsxwriter") as writer:
    for sheet_name, csv_path in deg_paths.items():
        df = pd.read_csv(csv_path)
        df.to_excel(writer, sheet_name=sheet_name, index=False)
        print(f"Wrote {sheet_name}: {df.shape}")

print(f"\nSaved Excel workbook to:\n{out_path}")

Wrote Mic_Discovery48: (8851, 8)
Wrote Oli_Discovery48: (13838, 8)
Wrote Mic_Validation48: (2413, 8)
Wrote Oli_Validation48: (3022, 8)

Saved Excel workbook to:
/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/DE_under_matched_Neuropathology.xlsx


In [ ]:
/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables